# One-seed Muon/MuonClip weight-quotient search

Analyze the verified final-100 model-only checkpoints after the
baseline run. The midpoint ECS is selected once from the raw
`fix_fingers=clip_xmax` WeightWatcher row at each checkpoint and is
then frozen before any quotient transformation. Every candidate is
materialized as the full-shape rectangular-diagonal canonical
representative of its diagnostic `O(out) x O(in)` orbit and passed
to WeightWatcher twice: `fix_fingers=False` and
`fix_fingers="clip_xmax"`.

This notebook tests five nuisance models rather than asserting that
an unknown Muon history has an exact non-trivial quotient. FC3 is
retained as a ten-mode auxiliary-AdamW control; FC1 and FC2 are the
Muon/MuonClip matrices. This orthogonal orbit is a spectral
diagnostic quotient, not a ReLU-network reparameterization
symmetry; transformed models are never evaluated for forward
accuracy. Entry-randomization diagnostics remain enabled for API
compatibility but are labelled gauge-dependent; only the ESD fit
is an orbit invariant. Completed checkpoint/profile groups are
atomically resumable under a code, parameter and run-fingerprint
identity check.


In [ ]:
# Papermill parameters. Override these values in an injected cell.
RUN_ROOT = ""
OUTPUT_ROOT = ""
CHECKPOINT_CACHE_ROOT = ""
CONFIG_PATH = ""
PROFILE = "pilot_1000_epochs"
PROTOCOL_SLUG = ""
SEEDS = [1337, 2027, 31415]
CHECKPOINT_PAYLOAD_CACHE_SIZE = 24
SHOW_PLOTS = True
REQUIRE_ARTIFACTS = True
ALLOW_TEMPORARY_LONG_RUN = False
SEED = 1337
RUN_PARAMETER_SCANS = True
OPTIMIZER_SLUGS = ["muon", "muonclip_rms"]

LAYERS = ["fc1.weight", "fc2.weight", "fc3.weight"]
MAXIMUM_CHECKPOINTS = 100
ANALYSIS_EPOCH_STRIDE = 1
RESUME_PARTIAL_RESULTS = True

# Exact existing WeightWatcher contract. Both fits analyze the same transformed model.
WW_MIN_EVALS = 8
WW_MAX_EVALS = None
WW_MAX_FINGERS = 10
WW_SVD_METHOD = "accurate"

# The primary profile is used by the three-seed notebook. The one-seed notebook
# defaults to the explicit scan profiles. Expand these lists rather than hiding
# additional optimization inside a helper.
UNIFORM_SINGULAR_PRIMARY = [
    {"profile_id": "mu_fraction_0p50", "shift_fraction": 0.50},
]
UNIFORM_SINGULAR_SCAN = [
    {"profile_id": "mu_fraction_0p25", "shift_fraction": 0.25},
    {"profile_id": "mu_fraction_0p50", "shift_fraction": 0.50},
    {"profile_id": "mu_fraction_0p75", "shift_fraction": 0.75},
]
GRAM_RIDGE_PRIMARY = [
    {"profile_id": "tau_fraction_0p50", "tau_fraction": 0.50},
]
GRAM_RIDGE_SCAN = [
    {"profile_id": "tau_fraction_0p25", "tau_fraction": 0.25},
    {"profile_id": "tau_fraction_0p50", "tau_fraction": 0.50},
    {"profile_id": "tau_fraction_0p75", "tau_fraction": 0.75},
]
BLOCKWISE_PRIMARY = [
    {"profile_id": "blocks_2_shift_0p50", "block_count": 2, "shift_fraction": 0.50},
]
BLOCKWISE_SCAN = [
    {"profile_id": "blocks_2_shift_0p25", "block_count": 2, "shift_fraction": 0.25},
    {"profile_id": "blocks_2_shift_0p50", "block_count": 2, "shift_fraction": 0.50},
    {"profile_id": "blocks_3_shift_0p50", "block_count": 3, "shift_fraction": 0.50},
]
FESHBACH_PRIMARY = [
    {"profile_id": "ridge_ratio_1em2", "regularization_ratio": 1.0e-2, "minimum_anchor_gap_ratio": 1.0e-6},
]
FESHBACH_SCAN = [
    {"profile_id": "ridge_ratio_1em4", "regularization_ratio": 1.0e-4, "minimum_anchor_gap_ratio": 1.0e-6},
    {"profile_id": "ridge_ratio_1em2", "regularization_ratio": 1.0e-2, "minimum_anchor_gap_ratio": 1.0e-6},
    {"profile_id": "ridge_ratio_1em1", "regularization_ratio": 1.0e-1, "minimum_anchor_gap_ratio": 1.0e-6},
]
RECTANGULAR_D_PRIMARY = [
    {"profile_id": "bulk_fraction_1p00", "minimum_noise_modes": 8, "noise_bulk_fraction": 1.00, "minimum_relative_separation": 0.01, "denominator_ridge": 1.0e-12},
]
RECTANGULAR_D_SCAN = [
    {"profile_id": "bulk_fraction_0p50", "minimum_noise_modes": 8, "noise_bulk_fraction": 0.50, "minimum_relative_separation": 0.01, "denominator_ridge": 1.0e-12},
    {"profile_id": "bulk_fraction_0p75", "minimum_noise_modes": 8, "noise_bulk_fraction": 0.75, "minimum_relative_separation": 0.01, "denominator_ridge": 1.0e-12},
    {"profile_id": "bulk_fraction_1p00", "minimum_noise_modes": 8, "noise_bulk_fraction": 1.00, "minimum_relative_separation": 0.01, "denominator_ridge": 1.0e-12},
]
CALIBRATED_SHRINKER_PRIMARY = [
    {"profile_id": "MP_scale_1p00", "minimum_noise_modes": 8, "noise_scale_multiplier": 1.00},
]
CALIBRATED_SHRINKER_SCAN = [
    {"profile_id": "MP_scale_0p75", "minimum_noise_modes": 8, "noise_scale_multiplier": 0.75},
    {"profile_id": "MP_scale_1p00", "minimum_noise_modes": 8, "noise_scale_multiplier": 1.00},
    {"profile_id": "MP_scale_1p25", "minimum_noise_modes": 8, "noise_scale_multiplier": 1.25},
]


In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from functools import lru_cache
import inspect
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "baseline" / "rg_baselines").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find baseline/rg_baselines. Launch Jupyter from a clone of "
        "CalculatedContent/rg_optimizers."
    )
BASELINE_ROOT = REPO_ROOT / "baseline"
EXPERIMENT_ROOT = BASELINE_ROOT / "experiments" / "mnist_mlp3_tangent_rg"
if str(BASELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(BASELINE_ROOT))

default_root = os.environ.get(
    "RG_MNIST_TANGENT_ROOT", "/tmp/rg-mnist-mlp3-tangent-rg"
)
RUN_ROOT_PATH = Path(RUN_ROOT or default_root).expanduser().resolve()

default_checkpoint_cache_root = os.environ.get(
    "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT",
    "/tmp/rg-mnist-mlp3-tangent-checkpoints",
)
CHECKPOINT_CACHE_ROOT_PATH = Path(
    CHECKPOINT_CACHE_ROOT or default_checkpoint_cache_root
).expanduser().resolve()

def _suite_name_from_profile():
    if str(PROTOCOL_SLUG).strip():
        return str(PROTOCOL_SLUG).strip()
    candidate = (
        Path(CONFIG_PATH).expanduser()
        if str(CONFIG_PATH).strip()
        else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
    )
    if candidate.is_file():
        if candidate.suffix.lower() == ".json":
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            value = payload.get("protocol", {}).get("suite_name")
            if value:
                return str(value)
        else:
            for line in candidate.read_text(encoding="utf-8").splitlines():
                stripped = line.strip()
                if stripped.startswith("suite_name:"):
                    return stripped.split(":", 1)[1].strip().strip("'\"")
    fallback = {
        "smoke": "mnist_mlp3_tangent_rg_v1_smoke",
        "pilot_1000_epochs": "mnist_mlp3_tangent_rg_v1_pilot1000",
        "long_horizon_10000_epochs": "mnist_mlp3_tangent_rg_v1_reference10000",
    }
    if PROFILE not in fallback:
        raise FileNotFoundError(
            f"Cannot derive suite_name for PROFILE={PROFILE!r}; set CONFIG_PATH "
            "or PROTOCOL_SLUG explicitly."
        )
    return fallback[PROFILE]

PROTOCOL_SLUG = _suite_name_from_profile()
OUTPUT_ROOT_PATH = Path(
    OUTPUT_ROOT or RUN_ROOT_PATH / PROTOCOL_SLUG / "notebook_outputs"
).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)

SEEDS = tuple(int(seed) for seed in SEEDS)
if SEEDS != (1337, 2027, 31415):
    print("WARNING: this is not the preregistered three-seed tuple:", SEEDS)

print("repository:", REPO_ROOT)
print("run root:", RUN_ROOT_PATH)
print("tail checkpoint cache root:", CHECKPOINT_CACHE_ROOT_PATH)
print("effective suite:", PROTOCOL_SLUG)
print("output root:", OUTPUT_ROOT_PATH)
print("seeds:", SEEDS)


In [ ]:
from rg_baselines.statistics import summarize_numeric_metrics
from rg_baselines.tangent_rg import powerlaw_fit, trace_log


from rg_baselines.tangent_rg import (
    AdamWProfile,
    MuonClipRMSProfile,
    MuonProfile,
    TangentRGConfig,
    load_analysis_checkpoint,
    list_analysis_checkpoints,
    list_capture_files,
    load_step_capture,
    replay_calibrated_step,
)
from rg_baselines.tangent_rg.checkpoints import load_verified_tail_checkpoint_refs
from rg_baselines.tangent_rg.protocol import tail_checkpoint_epochs
from rg_baselines.tangent_rg import nulls, polar, single_checkpoint, stiefel, two_checkpoint


import copy
import hashlib

import torch

from rg_baselines.model import MLP3
from rg_baselines.statistics import summarize_numeric_metrics
from rg_baselines.tangent_rg import weight_quotients, weightwatcher_fit
from rg_baselines.tangent_rg.weightwatcher_fit import (
    analyze_weightwatcher_dual,
    validate_weightwatcher_measurement,
)


## Weight-state quotient hypotheses

**`operator_kind`: `midpoint_ECS_weight_representatives_fit_by_dual_WeightWatcher`**

**`map_definition`: `P_mid(W_raw) is frozen; each method chooses the rectangular-diagonal canonical representative of [W_q] under O(m)xO(n); WeightWatcher fits its Gram energies.`**

**Identifiability caveat.** Muon polar-normalizes an update before adding it to W. The five maps below are falsifiable inverse models, not an exact quotient by arbitrary unknown semiorthogonal histories.

These strings are persisted with every result row. A visually useful
spectrum does not change the identity of the map that produced it.


In [ ]:
def require_path(path, *, description="artifact"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Run the prerequisite numbered notebook or set RUN_ROOT / "
            "OUTPUT_ROOT to the completed protocol directory."
        )
    return path


def first_existing(directory, names, *, description):
    directory = Path(directory)
    candidates = [directory / name for name in names]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {description} beneath {directory}. Expected one of:\n"
        + "\n".join(f"  - {path}" for path in candidates)
    )


def resolve_protocol_root():
    direct = RUN_ROOT_PATH / PROTOCOL_SLUG
    return direct if direct.is_dir() else RUN_ROOT_PATH


def resolve_arm_dir(optimizer_slug):
    protocol = resolve_protocol_root()
    candidates = [
        protocol / optimizer_slug,
        protocol / "results" / optimizer_slug,
        RUN_ROOT_PATH / optimizer_slug,
        RUN_ROOT_PATH / "results" / optimizer_slug,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    if REQUIRE_ARTIFACTS:
        raise FileNotFoundError(
            f"No completed {optimizer_slug!r} arm was found. Checked:\n"
            + "\n".join(f"  - {path}" for path in candidates)
        )
    return candidates[0]


def resolve_seed_dir(optimizer_slug, seed):
    arm = resolve_arm_dir(optimizer_slug)
    candidates = [
        arm / f"seed_{int(seed)}",
        arm / f"seed_{int(seed):05d}",
        arm / "seeds" / f"seed_{int(seed)}",
        arm / "seeds" / f"seed_{int(seed):05d}",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Missing seed directory for optimizer={optimizer_slug}, seed={seed}. "
        f"Checked {candidates}."
    )


def validate_run_identity(seed_dir, *, optimizer_slug, seed):
    seed_dir = Path(seed_dir)
    manifest = json.loads(
        require_path(seed_dir / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    resolved = json.loads(
        require_path(seed_dir / "resolved_config.json", description="resolved config")
        .read_text(encoding="utf-8")
    )
    completion = json.loads(
        require_path(seed_dir / "run_complete.json", description="completion marker")
        .read_text(encoding="utf-8")
    )
    config = dict(resolved.get("config", resolved))
    checks = {
        "manifest suite": (manifest.get("suite_name"), PROTOCOL_SLUG),
        "resolved suite": (config.get("suite_name"), PROTOCOL_SLUG),
        "manifest optimizer": (manifest.get("optimizer"), optimizer_slug),
        "resolved optimizer": (config.get("optimizer"), optimizer_slug),
        "completion optimizer": (completion.get("optimizer"), optimizer_slug),
        "manifest seed": (manifest.get("seed"), int(seed)),
        "resolved seed": (config.get("seed"), int(seed)),
        "completion seed": (completion.get("seed"), int(seed)),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in checks.items()
        if str(observed) != str(expected)
    ]
    fingerprints = {
        str(manifest.get("protocol_fingerprint", "")),
        str(resolved.get("protocol_fingerprint", "")),
        str(completion.get("protocol_fingerprint", "")),
    }
    if "" in fingerprints or len(fingerprints) != 1:
        mismatches.append(
            "manifest/resolved/completion protocol fingerprints are missing or unequal"
        )
    if not bool(completion.get("completed", False)):
        mismatches.append("run_complete.json does not declare completed=true")
    try:
        resolved_epochs = int(config["epochs"])
        completion_epochs = int(completion["epochs"])
        completion_step = int(completion["global_step"])
        best_validation_epoch = int(completion["best_validation_epoch"])
        analysis_plan = dict(resolved["analysis_plan"])
        plan_steps_per_epoch = int(analysis_plan["steps_per_epoch"])
        plan_total_steps = int(analysis_plan["total_steps"])
    except (KeyError, TypeError, ValueError) as error:
        mismatches.append(
            "resolved/completion final-horizon metadata is missing or invalid: "
            f"{type(error).__name__}: {error}"
        )
    else:
        if resolved_epochs < 1 or plan_steps_per_epoch < 1:
            mismatches.append("resolved epochs and steps_per_epoch must be positive")
        if plan_total_steps != resolved_epochs * plan_steps_per_epoch:
            mismatches.append(
                "resolved analysis_plan total_steps does not equal "
                "epochs * steps_per_epoch"
            )
        if completion_epochs != resolved_epochs:
            mismatches.append(
                f"completion epochs={completion_epochs} != resolved epochs={resolved_epochs}"
            )
        if completion_step != plan_total_steps:
            mismatches.append(
                f"completion global_step={completion_step} != resolved "
                f"analysis_plan total_steps={plan_total_steps}"
            )
        if not 0 <= best_validation_epoch <= resolved_epochs:
            mismatches.append(
                f"best_validation_epoch={best_validation_epoch} is outside "
                f"[0, {resolved_epochs}]"
            )
    if mismatches:
        raise RuntimeError(
            f"Run identity/provenance mismatch beneath {seed_dir}:\n  - "
            + "\n  - ".join(mismatches)
        )
    return manifest, resolved, completion


def validate_cross_run_provenance(manifests):
    manifests = list(manifests)
    if not manifests:
        raise RuntimeError("No manifests supplied for cross-run provenance audit")
    invariant_fields = (
        "suite_name", "dataset", "model", "initialization", "normalization",
        "train_indices_sha256", "validation_indices_sha256",
        "test_monitoring_only", "analysis_plan", "device", "software_versions",
        "determinism_settings",
    )
    disagreements = []
    for field in invariant_fields:
        serialized = {
            json.dumps(item.get(field), sort_keys=True, default=str)
            for item in manifests
        }
        if len(serialized) != 1:
            disagreements.append(field)
    if disagreements:
        raise RuntimeError(
            "Matched arms disagree on frozen run provenance fields: "
            + ", ".join(disagreements)
        )
    identities = {
        (str(item.get("optimizer")), int(item.get("seed"))) for item in manifests
    }
    expected = {
        (str(optimizer), int(seed))
        for optimizer in OPTIMIZER_SLUGS
        for seed in SEEDS
    } if "OPTIMIZER_SLUGS" in globals() else identities
    if identities != expected:
        raise RuntimeError(
            f"Manifest optimizer/seed grid is incomplete: observed={sorted(identities)}, "
            f"expected={sorted(expected)}"
        )
    return pd.DataFrame([
        {
            "optimizer": item.get("optimizer"),
            "seed": item.get("seed"),
            "device": item.get("device"),
            "software_versions": json.dumps(
                item.get("software_versions"), sort_keys=True, default=str
            ),
            "determinism_settings": json.dumps(
                item.get("determinism_settings"), sort_keys=True, default=str
            ),
            "pooling_compatibility_policy": (
                "headline pooling requires identical device, software versions, "
                "determinism settings, and scientific invariants across all runs"
            ),
        }
        for item in manifests
    ])


def record_dict(value):
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, dict):
        return dict(value)
    if hasattr(value, "__dict__"):
        return dict(vars(value))
    raise TypeError(f"Cannot convert {type(value).__name__} to an audit row")


def records_from_result(result):
    if result is None:
        return []
    if is_dataclass(result):
        return [record_dict(result)]
    if isinstance(result, dict):
        if "operator_kind" in result:
            return [dict(result)]
        rows = []
        for value in result.values():
            rows.extend(records_from_result(value))
        return rows
    if isinstance(result, (tuple, list)):
        rows = []
        for value in result:
            rows.extend(records_from_result(value))
        return rows
    return [record_dict(result)]


def spectrum_from_record(row):
    for name in (
        "spectrum", "eigenvalues", "singular_values", "rates",
        "positive_spectrum", "gram_spectrum",
    ):
        if name in row:
            values = np.asarray(row[name], dtype=float).reshape(-1)
            return values[np.isfinite(values) & (values > 0.0)]
    raise KeyError(
        "Operator record contains no recognized positive spectrum field. "
        f"Available fields: {sorted(row)}"
    )


def positive_spectrum(values, *, minimum_count=2):
    sample = np.asarray(values, dtype=float).reshape(-1)
    sample = sample[np.isfinite(sample) & (sample > 0.0)]
    sample = np.sort(sample)
    if sample.size < int(minimum_count):
        raise ValueError(
            f"Need at least {minimum_count} finite positive spectral values; "
            f"found {sample.size}."
        )
    return sample


def fit_spectrum_with_trace(
    values,
    *,
    operator_kind,
    map_definition,
    spectrum_kind,
    metadata,
    top_k_values=(0, 1, 2, 3, 4, 5),
    minimum_tail=8,
):
    # Fit amplitudes once, transform that fit to energy, and audit trace-log.
    # The power-law package is never called independently on squared values.
    # Trace-log uses squared values at the amplitude fit's independent rank.

    if str(spectrum_kind) != "amplitude":
        raise ValueError(
            "fit_spectrum_with_trace accepts operator amplitudes only; "
            "energy rows are produced by the exact amplitude-to-energy transform."
        )

    sample = positive_spectrum(values)
    feasible_top_k = tuple(
        int(value) for value in top_k_values if int(value) <= sample.size - 2
    )
    if not feasible_top_k or feasible_top_k[0] != 0:
        feasible_top_k = (0,)
    amplitude_fits = powerlaw_fit.fit_clipping_sensitivity(
        sample,
        top_k_values=feasible_top_k,
        minimum_tail=int(minimum_tail),
        operator_kind=str(operator_kind),
        map_definition=str(map_definition),
        spectrum_kind="amplitude",
        metadata=dict(metadata),
    )
    energy_rows = [
        powerlaw_fit.amplitude_fit_to_energy(row)
        for row in amplitude_fits.to_dict(orient="records")
    ]
    fits = pd.concat(
        [amplitude_fits, pd.DataFrame(energy_rows)],
        ignore_index=True,
        sort=False,
    )
    primary = amplitude_fits.loc[amplitude_fits["clip_top_k"].eq(0)].iloc[0]
    energy = sample ** 2
    trace_row = {
        **dict(metadata),
        "operator_kind": str(operator_kind),
        "map_definition": str(map_definition),
        "spectrum_kind": "energy_derived_from_amplitude",
        "support_rank_source": "powerlaw.Fit package-selected xmin tail count",
        "support_selected_from_same_trace_log": False,
        "support_rank": int(primary.get("n_tail", 0)),
        "trace_log_total": np.nan,
        "trace_log_per_eval": np.nan,
        "lambda_cut_scaled": np.nan,
        "trace_status": "fit_has_no_supported_tail",
    }
    rank = int(primary.get("n_tail", 0))
    if rank > 0:
        evaluated = trace_log.trace_log_at_rank(
            energy,
            rank=min(rank, energy.size),
            normalization_dimension=float(energy.size),
            rank_source="powerlaw.Fit package-selected xmin tail count",
        )
        trace_row.update(evaluated)
        trace_row["trace_status"] = "ok"
    return fits, pd.DataFrame([trace_row])


def save_analysis_frames(method_slug, *, operators, fits, traces):
    destination = OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
    destination.mkdir(parents=True, exist_ok=True)
    required_identity = {
        "optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"
    }
    for label, frame in (("operators", operators), ("fits", fits), ("traces", traces)):
        missing = required_identity - set(frame.columns)
        if missing:
            raise RuntimeError(
                f"{method_slug} {label} lack analysis provenance: {sorted(missing)}"
            )
        if frame[list(required_identity)].isna().any().any():
            raise RuntimeError(f"{method_slug} {label} contain null analysis provenance")
    identity_rows = fits[
        ["optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"]
    ].drop_duplicates()
    duplicate_fingerprints = (
        identity_rows.groupby(["optimizer", "seed"], dropna=False)[
            "protocol_fingerprint"
        ].nunique()
    )
    if (duplicate_fingerprints != 1).any():
        raise RuntimeError(
            f"{method_slug} has multiple protocol fingerprints for one optimizer/seed"
        )
    fingerprint_grid = {
        f"{row.optimizer}:{int(row.seed)}": str(row.protocol_fingerprint)
        for row in identity_rows.itertuples(index=False)
    }
    expected_grid_count = identity_rows[["optimizer", "seed"]].drop_duplicates().shape[0]
    if len(fingerprint_grid) != expected_grid_count:
        raise RuntimeError(f"{method_slug} fingerprint-grid keys are not unique")
    provenance_manifest = {
        "schema_version": 1,
        "suite_name": str(PROTOCOL_SLUG),
        "method_slug": str(method_slug),
        "optimizer_seed_protocol_fingerprints": dict(sorted(fingerprint_grid.items())),
        "source_artifact_kinds": sorted(
            identity_rows["source_artifact_kind"].astype(str).unique().tolist()
        ),
        "operator_row_count": int(len(operators)),
        "fit_row_count": int(len(fits)),
        "trace_row_count": int(len(traces)),
    }
    provenance_manifest["analysis_contract_tokens"] = sorted(
        fits["analysis_contract_token"].dropna().astype(str).unique().tolist()
        if "analysis_contract_token" in fits.columns
        else []
    )
    operators.to_csv(destination / "operator_rows.csv", index=False)
    fits.to_csv(destination / "powerlaw_fits.csv", index=False)
    traces.to_csv(destination / "trace_log_independent_support.csv", index=False)
    (destination / "method_provenance.json").write_text(
        json.dumps(provenance_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return destination


def save_spectrum_ccdf_gallery(
    spectral_arrays,
    *,
    method_slug,
    maximum_panels=24,
):
    # Save bounded log-log PDF/CCDF diagnostics for positive amplitudes.
    gallery = OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "spectrum_pdf_ccdf"
    gallery.mkdir(parents=True, exist_ok=True)
    rows = []
    for index, (key, raw) in enumerate(sorted(spectral_arrays.items())):
        if index >= int(maximum_panels):
            break
        sample = positive_spectrum(raw)
        x, ccdf = powerlaw_fit.empirical_ccdf(sample)
        fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
        if sample[0] < sample[-1]:
            bins = np.geomspace(sample[0], sample[-1], min(50, max(8, sample.size // 3)))
            axes[0].hist(sample, bins=bins, density=True, histtype="step", linewidth=1.8)
        else:
            axes[0].scatter(sample, np.ones_like(sample), s=15)
        axes[1].step(x, ccdf, where="post", linewidth=1.8)
        for axis in axes:
            axis.set_xscale("log")
            axis.set_yscale("log")
            axis.grid(alpha=0.2)
        axes[0].set(xlabel="amplitude b", ylabel="density", title="PDF")
        axes[1].set(xlabel="amplitude b", ylabel="P(B >= b)", title="CCDF")
        fig.suptitle(str(key), fontsize=8)
        fig.tight_layout()
        safe = "".join(character if character.isalnum() or character in "-_" else "_" for character in str(key))
        path = gallery / f"{index:03d}_{safe[:160]}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        if SHOW_PLOTS and index < 3:
            plt.show()
        else:
            plt.close(fig)
        rows.append({"spectrum_key": str(key), "n_positive": int(sample.size), "figure": str(path)})
    index_frame = pd.DataFrame(rows)
    index_frame.to_csv(gallery / "index.csv", index=False)
    return index_frame


def plot_fit_alpha_ci(fits, *, method_slug, title):
    usable = fits.copy()
    if "fit_ok" in usable:
        usable = usable[boolean_series(usable["fit_ok"])]
    if "spectrum_kind" in usable:
        energy = usable[
            usable["spectrum_kind"].astype(str).eq("energy_derived_from_amplitude")
        ]
        if not energy.empty:
            usable = energy
    primary = usable[usable["clip_top_k"].eq(0)] if "clip_top_k" in usable else usable
    if primary.empty:
        raise RuntimeError(
            f"{method_slug}: no successful preregistered raw fits; inspect powerlaw_fits.csv"
        )
    if "state_index" not in primary:
        primary["state_index"] = 0
    groups = tuple(
        name
        for name in (
            "optimizer", "layer", "method", "null_kind", "pair_stride",
            "epsilon", "evidence_role",
        )
        if name in primary
    )
    if not groups:
        primary["method"] = str(method_slug)
        groups = ("method",)
    return plot_seed_ci(
        primary,
        x="state_index",
        metric="alpha",
        groups=groups,
        title=title,
        ylabel="Power-law density exponent alpha",
        reference=2.0,
        allow_incomplete=True,
        incomplete_output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
            / "incomplete_alpha_ci_groups.csv"
        ),
        output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "alpha_95ci.png"
        ),
    )


def call_supported(function, /, *args, **kwargs):
    signature = inspect.signature(function)
    if any(
        parameter.kind is inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    ):
        return function(*args, **kwargs)
    supported = {key: value for key, value in kwargs.items() if key in signature.parameters}
    return function(*args, **supported)


def boolean_series(values):
    if getattr(values, "dtype", None) == bool:
        return values
    return values.astype(str).str.strip().str.lower().isin({"1", "true", "yes"})


def ci_summary(
    frame,
    *,
    groups,
    metrics,
    allow_incomplete=False,
    incomplete_output_path=None,
    return_incomplete=False,
):
    missing = set((*groups, *metrics, "seed")) - set(frame.columns)
    if missing:
        raise ValueError(f"CI input is missing columns: {sorted(missing)}")
    # Repeated layers/checkpoints/probes are not independent replicates.  First
    # collapse every declared group to one value per complete training seed.
    replicate = (
        frame.groupby([*groups, "seed"], as_index=False, dropna=False)[list(metrics)]
        .mean(numeric_only=True)
    )
    summary = summarize_numeric_metrics(
        replicate,
        group_columns=tuple(groups),
        metrics=tuple(metrics),
        confidence=0.95,
    )
    if summary.empty:
        raise RuntimeError("Confidence-interval summary is empty after seed aggregation")
    incomplete = summary[pd.to_numeric(summary["n"], errors="coerce") != len(SEEDS)]
    if not incomplete.empty:
        if incomplete_output_path is not None:
            incomplete_output_path = Path(incomplete_output_path)
            incomplete_output_path.parent.mkdir(parents=True, exist_ok=True)
            incomplete.to_csv(incomplete_output_path, index=False)
        if allow_incomplete:
            print(
                "WARNING: dropping incomplete CI identities from the mean/band; "
                "faint individual-seed traces remain visible.\n"
                + incomplete[
                    [name for name in (*groups, "metric", "n") if name in incomplete]
                ].to_string(index=False)
            )
        else:
            identity = [name for name in (*groups, "metric", "n") if name in incomplete]
            raise RuntimeError(
                "Every confidence-interval row requires exactly the preregistered "
                f"{len(SEEDS)} complete seeds. Incomplete identities:\n"
                + incomplete[identity].to_string(index=False)
            )
    complete = summary[pd.to_numeric(summary["n"], errors="coerce") == len(SEEDS)].copy()
    if return_incomplete:
        return complete, incomplete.copy()
    return complete


def plot_seed_ci(
    frame,
    *,
    x,
    metric,
    groups,
    title,
    ylabel,
    reference=None,
    output_path=None,
    allow_incomplete=False,
    incomplete_output_path=None,
):
    groups = tuple(groups)
    if allow_incomplete and incomplete_output_path is None and output_path is not None:
        output_path_for_report = Path(output_path)
        incomplete_output_path = output_path_for_report.with_name(
            output_path_for_report.stem + "_incomplete_ci_groups.csv"
        )
    summary, incomplete = ci_summary(
        frame,
        groups=(*groups, x),
        metrics=(metric,),
        allow_incomplete=allow_incomplete,
        incomplete_output_path=incomplete_output_path,
        return_incomplete=True,
    )
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    if not groups:
        frame = frame.copy()
        frame["series"] = "all"
        groups = ("series",)
    for identity, group in frame.groupby(list(groups), dropna=False):
        identity = identity if isinstance(identity, tuple) else (identity,)
        label = ", ".join(f"{key}={value}" for key, value in zip(groups, identity))
        for _, seed_frame in group.groupby("seed"):
            ordered = (
                seed_frame.groupby(x, as_index=False, dropna=False)[metric]
                .mean(numeric_only=True)
                .sort_values(x)
            )
            ax.plot(ordered[x], ordered[metric], alpha=0.16, linewidth=0.9)
        selected = summary.copy()
        for key, value in zip(groups, identity):
            selected = selected[selected[key].astype(str) == str(value)]
        selected = selected[selected["metric"] == metric].sort_values(x)
        if selected.empty:
            pass
        else:
            xv = selected[x].to_numpy(dtype=float)
            mean = selected["mean"].to_numpy(dtype=float)
            low = selected["ci_low"].to_numpy(dtype=float)
            high = selected["ci_high"].to_numpy(dtype=float)
            ax.plot(xv, mean, marker="o", linewidth=2.1, label=label)
            finite = np.isfinite(low) & np.isfinite(high)
            ax.fill_between(xv[finite], low[finite], high[finite], alpha=0.18)
        missing = incomplete.copy()
        for key, value in zip(groups, identity):
            missing = missing[missing[key].astype(str) == str(value)]
        missing = missing[missing["metric"] == metric]
        if not missing.empty:
            ax.scatter(
                missing[x].to_numpy(dtype=float),
                missing["mean"].to_numpy(dtype=float),
                marker="x", color="#555555", alpha=0.65, zorder=4,
            )
    if reference is not None:
        ax.axhline(float(reference), color="#333333", linestyle="--", linewidth=1.4)
    ax.set(xlabel=x, ylabel=ylabel, title=title)
    ax.set_xscale("symlog", linthresh=1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    return summary, fig


_VERIFIED_TAIL_CACHE_REFS = {}
_VERIFIED_TAIL_CHECKPOINT_IDENTITIES = {}
_VERIFIED_RUN_IDENTITIES = {}


def require_complete_seed(optimizer_slug, seed):
    seed_dir = resolve_seed_dir(optimizer_slug, seed)
    manifest, _, _ = validate_run_identity(
        seed_dir, optimizer_slug=optimizer_slug, seed=seed
    )
    _VERIFIED_RUN_IDENTITIES[(str(optimizer_slug), int(seed))] = {
        "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
        "source_seed_dir": str(Path(seed_dir).resolve()),
    }
    return seed_dir


def verified_run_fingerprint(optimizer_slug, seed):
    identity = _VERIFIED_RUN_IDENTITIES.get((str(optimizer_slug), int(seed)))
    if identity is None:
        raise RuntimeError(
            f"Run identity was not verified for optimizer={optimizer_slug}, seed={seed}"
        )
    return str(identity["protocol_fingerprint"])


def require_tail_checkpoint_cache(optimizer_slug, seed):
    # Establish expected identity from the separately completed run. Never
    # trust identity claimed only by the temporary cache itself.
    source_seed_dir = resolve_seed_dir(optimizer_slug, seed)
    manifest, resolved, completion = validate_run_identity(
        source_seed_dir, optimizer_slug=optimizer_slug, seed=seed
    )
    _VERIFIED_RUN_IDENTITIES[(str(optimizer_slug), int(seed))] = {
        "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
        "source_seed_dir": str(Path(source_seed_dir).resolve()),
    }
    resolved_values = dict(resolved.get("config", resolved))
    if "epochs" not in resolved_values:
        raise KeyError(f"Resolved run config lacks epochs: {source_seed_dir}")
    expected_epochs = tail_checkpoint_epochs(int(resolved_values["epochs"]))
    recorded_cache_root = Path(
        resolved_values.get(
            "tail_checkpoint_cache_root",
            "/tmp/rg-mnist-mlp3-tangent-checkpoints",
        )
    ).expanduser().resolve()
    if recorded_cache_root != CHECKPOINT_CACHE_ROOT_PATH:
        raise RuntimeError(
            "Notebook CHECKPOINT_CACHE_ROOT disagrees with the completed run: "
            f"notebook={CHECKPOINT_CACHE_ROOT_PATH}, recorded={recorded_cache_root}. "
            "Set CHECKPOINT_CACHE_ROOT (or "
            "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT) to the recorded cache root."
        )
    temporary_root = Path("/tmp").resolve()
    if (
        recorded_cache_root == temporary_root
        or not recorded_cache_root.is_relative_to(temporary_root)
    ):
        raise RuntimeError(
            f"Tail checkpoint cache must be a safe child of /tmp: {recorded_cache_root}"
        )
    cache_seed_dir = (
        recorded_cache_root
        / PROTOCOL_SLUG
        / str(optimizer_slug)
        / f"seed_{int(seed)}"
    ).resolve()
    refs = tuple(
        load_verified_tail_checkpoint_refs(
            cache_seed_dir,
            expected_suite_name=PROTOCOL_SLUG,
            expected_optimizer_name=str(optimizer_slug),
            expected_seed=int(seed),
            expected_fingerprint=str(manifest["protocol_fingerprint"]),
            expected_epochs=expected_epochs,
            validate_payloads=True,
        )
    )
    expected_count = min(100, int(resolved_values["epochs"]))
    if len(refs) != expected_count:
        raise RuntimeError(
            f"Verified cache count {len(refs)} != expected {expected_count}: "
            f"{cache_seed_dir}"
        )
    completed_epochs = int(completion.get("epochs", -1))
    completed_step = int(completion.get("global_step", -1))
    if completed_epochs != int(resolved_values["epochs"]) or completed_step < 1:
        raise RuntimeError(
            f"Completed run horizon is inconsistent beneath {source_seed_dir}"
        )
    steps_per_epoch, remainder = divmod(completed_step, completed_epochs)
    if remainder or steps_per_epoch < 1:
        raise RuntimeError(
            f"Completed step count is not an exact epoch grid beneath {source_seed_dir}"
        )
    expected_pairs = tuple(
        (int(epoch), int(epoch) * int(steps_per_epoch))
        for epoch in expected_epochs
    )
    observed_pairs = tuple((int(ref.epoch), int(ref.global_step)) for ref in refs)
    completion_cache_dir = completion.get("tail_checkpoint_cache_dir")
    if not isinstance(completion_cache_dir, str) or not completion_cache_dir.strip():
        raise RuntimeError(
            f"Completion marker lacks tail_checkpoint_cache_dir: {source_seed_dir}"
        )
    source_cache_checks = {
        "cache_dir": (
            str(Path(completion_cache_dir).expanduser().resolve()),
            str(cache_seed_dir),
        ),
        "checkpoint_count": (completion.get("tail_checkpoint_count"), expected_count),
        "first_epoch": (completion.get("tail_checkpoint_first_epoch"), expected_epochs[0]),
        "last_epoch": (completion.get("tail_checkpoint_last_epoch"), expected_epochs[-1]),
        "epoch_step_grid": (observed_pairs, expected_pairs),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in source_cache_checks.items()
        if observed != expected
    ]
    if mismatches:
        raise RuntimeError(
            "Tail cache disagrees with the persistent run completion marker:\n  - "
            + "\n  - ".join(mismatches)
        )
    resolved_cache_dir = cache_seed_dir.resolve()
    _VERIFIED_TAIL_CACHE_REFS[resolved_cache_dir] = refs
    for ref in refs:
        _VERIFIED_TAIL_CHECKPOINT_IDENTITIES[ref.path.resolve()] = {
            "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
            "optimizer": str(optimizer_slug),
            "seed": int(seed),
            "epoch": int(ref.epoch),
            "global_step": int(ref.global_step),
            "cache_seed_dir": str(resolved_cache_dir),
        }
    return resolved_cache_dir




def analysis_checkpoint_refs(seed_dir):
    cache_seed_dir = Path(seed_dir).resolve()
    refs = tuple(_VERIFIED_TAIL_CACHE_REFS.get(cache_seed_dir, ()))
    if len(refs) < 2:
        raise RuntimeError(
            "Tail checkpoint cache was not verified in this kernel, or contains "
            f"fewer than two states: {cache_seed_dir}. Call "
            "require_tail_checkpoint_cache(optimizer, seed); analysis notebooks "
            "never fall back to sparse run checkpoints or training."
        )
    return refs


def selected_checkpoint_pairs(seed_dir, *, maximum_pairs=None, stride=1):
    refs = analysis_checkpoint_refs(seed_dir)
    stride = int(stride)
    if stride < 1:
        raise ValueError("PAIR_STRIDE must be positive")
    all_pairs = list(zip(refs[:-stride], refs[stride:]))
    if not all_pairs:
        raise RuntimeError(f"No checkpoint pairs selected from {seed_dir}")
    budget = (
        len(all_pairs)
        if maximum_pairs is None
        else min(int(maximum_pairs), len(all_pairs))
    )
    if budget < 1:
        raise ValueError("maximum_pairs must be positive or None")
    if budget == len(all_pairs):
        indices = list(range(len(all_pairs)))
        complete_role = (
            "complete_verified_tail_adjacent_grid"
            if stride == 1
            else "complete_verified_tail_stride_grid"
        )
        rule = (
            "all chronological pairs from the verified tail checkpoint cache: "
            f"stride={stride}, selected_count={len(indices)}, "
            f"total_available={len(all_pairs)}"
        )
        roles = [complete_role for _ in indices]
    else:
        tail_count = min(max(2, budget // 3), budget)
        broad_budget = max(0, budget - tail_count)
        broad_limit = max(0, len(all_pairs) - tail_count)
        broad_indices = []
        if broad_budget and broad_limit:
            broad_indices = np.unique(
                np.rint(np.geomspace(1, broad_limit, num=broad_budget) - 1).astype(int)
            ).tolist()
            if 0 not in broad_indices:
                broad_indices.insert(0, 0)
        tail_indices = list(range(len(all_pairs) - tail_count, len(all_pairs)))
        indices = sorted(set(broad_indices + tail_indices))
        rule = (
            "deterministic log-index broad pair sample plus consecutive pair tail: "
            f"stride={stride}, requested={maximum_pairs}, selected_indices={indices}, "
            f"tail_count={tail_count}, total_available={len(all_pairs)}"
        )
        tail_start = len(all_pairs) - tail_count
        roles = [
            (
                "consecutive_tail"
                if stride == 1 and index >= tail_start
                else "stride_tail"
                if index >= tail_start
                else "log_index_broad"
            )
            for index in indices
        ]
    return [
        (
            all_pairs[index][0], all_pairs[index][1], rule, role,
        )
        for index, role in zip(indices, roles)
    ]


def selected_checkpoint_pairs_for_strides(seed_dir, *, strides, maximum_pairs=None):
    selected = []
    available_states = len(analysis_checkpoint_refs(seed_dir))
    for stride in tuple(dict.fromkeys(int(value) for value in strides)):
        if stride >= available_states:
            print(
                f"Skipping pair stride={stride}: verified cache has only "
                f"{available_states} states"
            )
            continue
        for previous, current, rule, role in selected_checkpoint_pairs(
            seed_dir, maximum_pairs=maximum_pairs, stride=stride
        ):
            selected.append((stride, previous, current, rule, role))
    if not selected:
        raise RuntimeError(f"No multi-spacing checkpoint pairs selected from {seed_dir}")
    return selected


def checkpoint_refs_at_epoch_stride(seed_dir, *, epoch_stride=1):
    refs = analysis_checkpoint_refs(seed_dir)
    stride = int(epoch_stride)
    if stride < 1:
        raise ValueError("ANALYSIS_EPOCH_STRIDE must be positive")
    if stride == 1:
        return refs
    selected = tuple(
        ref for ref in refs
        if int(ref.epoch) > 0 and int(ref.epoch) % stride == 0
    )
    if len(selected) < 2:
        raise RuntimeError(
            f"Epoch stride {stride} selected fewer than two checkpoints from "
            f"{seed_dir}: epochs={[int(ref.epoch) for ref in selected]}"
        )
    return selected


def selected_trajectory_matrices(
    seed_dir, *, layers, maximum_checkpoints, epoch_stride=1
):
    refs = checkpoint_refs_at_epoch_stride(
        seed_dir, epoch_stride=epoch_stride
    )
    budget = min(int(maximum_checkpoints), len(refs))
    if budget == len(refs):
        indices = list(range(len(refs)))
        roles = ["complete_verified_tail_state_grid" for _ in indices]
        rule = (
            "all chronological states from the verified tail checkpoint cache "
            "after exact positive epoch-modulus filtering: "
            f"epoch_stride={int(epoch_stride)}, selected_count={len(indices)}, "
            f"total_stride_eligible={len(refs)}"
        )
    else:
        tail_count = min(max(2, budget // 3), budget)
        broad_budget = max(0, budget - tail_count)
        broad_limit = max(0, len(refs) - tail_count)
        broad_indices = []
        if broad_budget and broad_limit:
            broad_indices = np.unique(
                np.rint(np.geomspace(1, broad_limit, num=broad_budget) - 1)
                .astype(int)
            ).tolist()
            if 0 not in broad_indices:
                broad_indices.insert(0, 0)
        tail_indices = list(range(len(refs) - tail_count, len(refs)))
        indices = sorted(set(broad_indices + tail_indices))
        tail_start = len(refs) - tail_count
        roles = [
            "consecutive_tail" if index >= tail_start else "log_index_broad"
            for index in indices
        ]
        rule = (
            f"deterministic log-index broad sample plus consecutive tail: "
            f"requested={maximum_checkpoints}, selected_indices={indices}, "
            f"tail_count={tail_count}, total_available={len(refs)}"
        )
    selected = [refs[index] for index in indices]
    if len(selected) < 2:
        raise RuntimeError(f"Need at least two trajectory states beneath {seed_dir}")
    for index, ref, role in zip(indices, selected, roles):
        for layer in layers:
            yield ref, str(layer), checkpoint_matrix(ref.path, str(layer)), rule, role


@lru_cache(maxsize=int(CHECKPOINT_PAYLOAD_CACHE_SIZE))
def _load_verified_checkpoint_payload_cached(
    source_text,
    expected_fingerprint,
    expected_optimizer,
    expected_seed,
    expected_epoch,
    expected_global_step,
):
    source = Path(source_text)
    payload = load_analysis_checkpoint(
        source, expected_fingerprint=str(expected_fingerprint)
    )
    checks = {
        "optimizer": (payload.get("optimizer"), str(expected_optimizer)),
        "seed": (payload.get("seed"), int(expected_seed)),
        "epoch": (payload.get("epoch"), int(expected_epoch)),
        "global_step": (payload.get("global_step"), int(expected_global_step)),
    }
    mismatches = [
        f"{field}: observed={observed!r}, expected={expected!r}"
        for field, (observed, expected) in checks.items()
        if str(observed) != str(expected)
    ]
    if mismatches:
        raise RuntimeError(
            f"Checkpoint payload disagrees with verified cache identity: {source}; "
            + "; ".join(mismatches)
        )
    return payload


def load_checkpoint_payload(path):
    source = require_path(path, description="verified tail checkpoint").resolve()
    expected = _VERIFIED_TAIL_CHECKPOINT_IDENTITIES.get(source)
    if expected is None:
        raise RuntimeError(
            f"Checkpoint was not admitted by the strict tail-cache verifier: {source}. "
            "Analysis never falls back to an arbitrary checkpoint path."
        )
    payload = _load_verified_checkpoint_payload_cached(
        str(source),
        str(expected["protocol_fingerprint"]),
        str(expected["optimizer"]),
        int(expected["seed"]),
        int(expected["epoch"]),
        int(expected["global_step"]),
    )
    match = re.fullmatch(
        r"analysis_epoch_(?P<epoch>\d+)_step_(?P<step>\d+)\.pt", source.name
    )
    if match is None:
        raise RuntimeError(f"Expected an immutable analysis-checkpoint filename: {source}")
    if (
        int(payload.get("epoch", -1)) != int(match.group("epoch"))
        or int(payload.get("global_step", -1)) != int(match.group("step"))
        or int(payload.get("epoch", -1)) != int(expected["epoch"])
        or int(payload.get("global_step", -1)) != int(expected["global_step"])
    ):
        raise RuntimeError(f"Checkpoint payload epoch/step disagrees with filename: {source}")
    return payload


def checkpoint_state_dict(payload):
    for key in ("model", "model_state_dict", "state_dict"):
        value = payload.get(key) if isinstance(payload, dict) else None
        if isinstance(value, dict):
            return value
    raise KeyError(
        "Checkpoint has no model/model_state_dict/state_dict mapping; "
        f"available keys={list(payload) if isinstance(payload, dict) else type(payload)}"
    )


def checkpoint_matrix(path, parameter_name):
    state = checkpoint_state_dict(load_checkpoint_payload(path))
    candidates = (
        parameter_name,
        parameter_name.removeprefix("model."),
        f"model.{parameter_name}",
    )
    for name in candidates:
        if name in state:
            value = state[name]
            if hasattr(value, "detach"):
                return value.detach().cpu().double().numpy()
            return np.asarray(value, dtype=np.float64)
    raise KeyError(
        f"Matrix {parameter_name!r} is absent from {path}; "
        f"available 2-D keys={[key for key, value in state.items() if getattr(value, 'ndim', 0) == 2]}"
    )


def checkpoint_step(payload, fallback):
    for key in ("global_step", "step", "optimizer_step"):
        if isinstance(payload, dict) and key in payload:
            return int(payload[key])
    return int(fallback)


def capture_payloads(seed_dir, *, maximum_captures=None):
    capture_root = Path(seed_dir) / "captures"
    require_path(capture_root, description="dense capture directory")
    paths = tuple(list_capture_files(capture_root))
    if not paths:
        raise FileNotFoundError(
            f"No dense capture files under {capture_root}. Ensure the resolved "
            "config includes burst anchors and rerun training."
        )
    if maximum_captures is not None:
        paths = paths[-int(maximum_captures):]
    manifest = json.loads(
        require_path(Path(seed_dir) / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    loaded = []
    for path in paths:
        payload = load_step_capture(
            path, expected_fingerprint=manifest["protocol_fingerprint"]
        )
        if str(payload.get("optimizer")) != str(manifest.get("optimizer")):
            raise RuntimeError(f"Capture optimizer disagrees with manifest: {path}")
        anchor_match = re.fullmatch(r"burst_epoch_(\d+)", path.parent.name)
        if (
            anchor_match is None
            or int(payload.get("anchor_epoch", -1)) != int(anchor_match.group(1))
        ):
            raise RuntimeError(f"Capture anchor epoch disagrees with directory: {path}")
        loaded.append((path, payload))
    return loaded


def capture_array(parameter_payload, key, *, required=True):
    value = parameter_payload.get(key)
    if value is None:
        if required:
            raise KeyError(
                f"Dense capture parameter is missing required field {key!r}; "
                f"available={sorted(parameter_payload)}"
            )
        return None
    if hasattr(value, "detach"):
        return value.detach().cpu().double().numpy()
    return np.asarray(value, dtype=np.float64)


def resolved_training_config(seed_dir):
    payload = json.loads(
        require_path(
            Path(seed_dir) / "resolved_config.json",
            description="resolved training config",
        ).read_text(encoding="utf-8")
    )
    values = dict(payload.get("config", payload))
    if not values:
        raise ValueError(f"Resolved config is empty in {seed_dir}")
    values["adamw"] = AdamWProfile(**dict(values.get("adamw", {})))
    muon_values = dict(values.get("muon", {}))
    if "parameter_names" in muon_values:
        muon_values["parameter_names"] = tuple(muon_values["parameter_names"])
    values["muon"] = MuonProfile(**muon_values)
    clip_values = dict(values.get("muonclip_rms", {}))
    if "parameter_names" in clip_values:
        clip_values["parameter_names"] = tuple(clip_values["parameter_names"])
    values["muonclip_rms"] = MuonClipRMSProfile(**clip_values)
    for name in (
        "explicit_analysis_epochs", "dense_burst_anchor_epochs",
        "capture_parameter_names",
    ):
        if name in values and isinstance(values[name], list):
            values[name] = tuple(values[name])
    config = TangentRGConfig(**values)
    config.validate()
    return config


EXPECTED_WEIGHT_LAYERS = ("fc1.weight", "fc2.weight", "fc3.weight")
EXPECTED_QUOTIENT_METHODS = (
    "gram_ridge",
    "blockwise_singular",
    "feshbach_downfolding",
    "rectangular_d_transform",
    "calibrated_mp_shrinker",
)
REFERENCE_METHODS = (
    "midpoint_ecs_control",
    "uniform_singular_translation",
)
FINAL_SPECTRUM_INDEX_COLUMNS = (
    "spectrum_key", "optimizer", "seed", "epoch", "global_step",
    "layer", "method", "profile_id",
)
WEIGHT_QUOTIENT_ANALYSIS_VERSION = "weight_quotient_notebooks_v2"
WEIGHT_QUOTIENT_RUNTIME_DEPENDENCY_SHA256 = hashlib.sha256(
    (
        WEIGHT_QUOTIENT_ANALYSIS_VERSION
        + inspect.getsource(weight_quotients)
        + inspect.getsource(weightwatcher_fit)
    ).encode("utf-8")
).hexdigest()


def _jsonable(value):
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return [_jsonable(item) for item in value.tolist()]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        converted = float(value)
        return converted if np.isfinite(converted) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def normalize_papermill_sequence(value, *, name):
    # Accept a sequence or the JSON-list string produced by Papermill -p.

    parsed = value
    if isinstance(value, str):
        try:
            parsed = json.loads(value.strip())
        except json.JSONDecodeError as error:
            raise ValueError(
                f"{name} must be a JSON list when supplied as a string; "
                f"observed {value!r}"
            ) from error
    if isinstance(parsed, (str, bytes)) or not isinstance(parsed, (list, tuple)):
        raise TypeError(
            f"{name} must be a list or tuple; observed {type(parsed).__name__}"
        )
    result = tuple(parsed)
    if not result:
        raise ValueError(f"{name} must not be empty")
    return result


def normalize_papermill_bool(value, *, name):
    # Do not treat the Papermill string "false" as truthy.

    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes"}:
            return True
        if normalized in {"false", "0", "no"}:
            return False
    raise TypeError(f"{name} must be a boolean; observed {value!r}")


WEIGHT_QUOTIENT_ANALYSIS_SETTINGS = {
    "schema_version": 1,
    "weightwatcher": {
        "min_evals": int(WW_MIN_EVALS),
        "max_evals": (
            None if WW_MAX_EVALS is None else int(WW_MAX_EVALS)
        ),
        "max_fingers": int(WW_MAX_FINGERS),
        "svd_method": str(WW_SVD_METHOD),
        "randomize": True,
        "primary_variant": "clip_xmax",
    },
    "expected_layers": list(EXPECTED_WEIGHT_LAYERS),
}
WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_JSON = json.dumps(
    _jsonable(WEIGHT_QUOTIENT_ANALYSIS_SETTINGS),
    sort_keys=True,
    separators=(",", ":"),
)
WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_SHA256 = hashlib.sha256(
    WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_JSON.encode("utf-8")
).hexdigest()


def stable_slug(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def parameter_profile(profile):
    values = dict(profile)
    profile_id = str(values.pop("profile_id"))
    return profile_id, values


def choose_profiles(primary, scan):
    selected = list(scan if bool(RUN_PARAMETER_SCANS) else primary)
    if not selected:
        raise ValueError("Every quotient method requires at least one parameter profile")
    profile_ids = [str(item.get("profile_id", "")) for item in selected]
    if any(not value for value in profile_ids) or len(set(profile_ids)) != len(profile_ids):
        raise ValueError(f"Parameter profile IDs must be non-empty and unique: {selected}")
    return selected


def stable_analysis_seed(*parts):
    digest = hashlib.sha256("|".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def spectrum_sha256(values):
    array = np.asarray(values, dtype="<f8").reshape(-1)
    array = np.sort(array[np.isfinite(array) & (array > 0.0)])
    array = np.ascontiguousarray(array)
    return hashlib.sha256(array.tobytes()).hexdigest()


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)
    return path


def atomic_npz(arrays, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **dict(arrays))
    temporary.replace(path)
    return path


def atomic_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(_jsonable(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)
    return path


def checkpoint_model(path):
    payload = load_checkpoint_payload(path)
    model = MLP3().to("cpu")
    model.load_state_dict(checkpoint_state_dict(payload), strict=True)
    model.eval()
    return model


def matrix_from_model(model, layer):
    value = dict(model.named_parameters())[str(layer)]
    return value.detach().cpu().double().numpy()


def replace_model_matrix(model, layer, matrix):
    parameter = dict(model.named_parameters())[str(layer)]
    candidate = torch.as_tensor(np.asarray(matrix), dtype=parameter.dtype)
    if tuple(candidate.shape) != tuple(parameter.shape):
        raise ValueError(
            f"Transformed {layer} shape {tuple(candidate.shape)} != {tuple(parameter.shape)}"
        )
    with torch.no_grad():
        parameter.copy_(candidate)


def model_layer_esd(model, layer):
    parameter = dict(model.named_parameters())[str(layer)]
    matrix = parameter.detach().float().cpu().numpy()
    values = np.linalg.svd(matrix, compute_uv=False) ** 2
    return np.sort(values[np.isfinite(values) & (values > 0.0)])


def numeric(row, name, default=np.nan):
    result = pd.to_numeric(pd.Series([row.get(name, default)]), errors="coerce").iloc[0]
    return float(result) if pd.notna(result) else float("nan")


def truthy_series(values):
    return values.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def midpoint_record(measurement, *, layer, metadata):
    rows = measurement.details[
        (measurement.details["layer"].astype(str) == str(layer))
        & (measurement.details["fit_variant"].astype(str) == "clip_xmax")
    ]
    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one clip_xmax row for {layer}; observed {len(rows)}"
        )
    row = rows.iloc[0]
    maximum_rank = int(np.asarray(measurement.esds[str(layer)]).size)
    pl_rank_value = numeric(row, "n_tail")
    detx_value = numeric(row, "detX_num", numeric(row, "detx_num"))
    if not np.isfinite(pl_rank_value) or not np.isfinite(detx_value):
        raise RuntimeError(
            f"Midpoint ECS unavailable for {layer}: n_tail={pl_rank_value}, "
            f"detX_num={detx_value}"
        )
    pl_rank = int(np.clip(round(pl_rank_value), 1, maximum_rank))
    detx_rank = int(np.clip(round(detx_value), 1, maximum_rank))
    midpoint = weight_quotients.midpoint_ecs_rank(
        pl_rank, detx_rank, maximum_rank=maximum_rank
    )
    return {
        **dict(metadata),
        "layer": str(layer),
        "maximum_rank": maximum_rank,
        "k_pl": pl_rank,
        "k_detx": detx_rank,
        "k_mid": midpoint,
        "midpoint_rule": "floor((max(1,k_pl)+k_detx)/2)",
        "midpoint_fit_variant": "clip_xmax",
        "midpoint_selected_before_quotient": True,
    }


def weightwatcher_rows(measurement, *, metadata, spectrum_hashes):
    frame = measurement.details.copy()
    for key, value in dict(metadata).items():
        frame[key] = value
    frame["transformed_spectrum_sha256"] = frame["layer"].map(spectrum_hashes)
    frame["weightwatcher_pair_contract"] = (
        "same_transformed_model_for_raw_and_fix_fingers_clip_xmax"
    )
    return frame


def unavailable_fit_rows(*, metadata, layer, reason, spectrum_hash=""):
    return pd.DataFrame(
        [
            {
                **dict(metadata),
                "layer": str(layer),
                "fit_variant": variant,
                "finger_policy": (
                    "none" if variant == "raw" else "fix_fingers=clip_xmax"
                ),
                "fit_ok": False,
                "weightwatcher_status": "quotient_unavailable_before_fit",
                "weightwatcher_error": str(reason),
                "alpha": np.nan,
                "ks_D": np.nan,
                "xmin": np.nan,
                "xmax": np.nan,
                "backend_xmax": np.nan,
                "n_tail": np.nan,
                "tail_decades": np.nan,
                "transformed_spectrum_sha256": str(spectrum_hash),
                "weightwatcher_pair_contract": (
                    "same_transformed_model_for_raw_and_fix_fingers_clip_xmax"
                ),
            }
            for variant in ("raw", "clip_xmax")
        ]
    )


CHECKPOINT_ID_COLUMNS = ("optimizer", "seed", "epoch", "global_step")


def checkpoint_identity(row):
    return (
        str(row["optimizer"]),
        int(row["seed"]),
        int(row["epoch"]),
        int(row["global_step"]),
    )


def checkpoint_profile_identity(row):
    return checkpoint_identity(row) + (str(row["profile_id"]),)


def _read_resume_csv(path):
    path = Path(path)
    if not bool(RESUME_PARTIAL_RESULTS) or not path.is_file():
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    if "analysis_code_sha256" not in frame:
        return pd.DataFrame()
    if set(frame["analysis_code_sha256"].astype(str)) != {
        WEIGHT_QUOTIENT_ANALYSIS_CODE_SHA256
    }:
        print("Ignoring stale partial table with a different analysis hash:", path)
        return pd.DataFrame()
    return frame


def load_resumable_tables(
    *, fit_path, secondary_path, contexts, method, profiles
):
    fits = _read_resume_csv(fit_path)
    secondary = _read_resume_csv(secondary_path)
    if fits.empty or secondary.empty:
        return pd.DataFrame(), pd.DataFrame(), set()
    active_fingerprints = {
        checkpoint_identity(row): str(row["protocol_fingerprint"])
        for _, row in contexts.iterrows()
    }
    active_analysis_settings = {
        checkpoint_identity(row): str(row["analysis_settings_sha256"])
        for _, row in contexts.iterrows()
    }
    active_keys = set(active_fingerprints)
    profile_definitions = {}
    for profile in profiles:
        profile_id, options = parameter_profile(profile)
        profile_definitions[profile_id] = json.dumps(
            _jsonable(options), sort_keys=True
        )

    def eligible(row):
        profile_id = str(row.get("profile_id", ""))
        return (
            checkpoint_identity(row) in active_keys
            and str(row.get("protocol_fingerprint", ""))
            == active_fingerprints.get(checkpoint_identity(row), "")
            and str(row.get("source_artifact_kind", ""))
            == "verified_final_100_tail_cache"
            and str(row.get("analysis_settings_sha256", ""))
            == active_analysis_settings.get(checkpoint_identity(row), "")
            and str(row.get("method", "")) == str(method)
            and profile_id in profile_definitions
            and str(row.get("parameter_profile", ""))
            == profile_definitions[profile_id]
        )

    fits = fits.loc[[eligible(row) for _, row in fits.iterrows()]].copy()
    secondary = secondary.loc[
        [eligible(row) for _, row in secondary.iterrows()]
    ].copy()
    complete_fit_keys = set()
    for identity, group in fits.groupby(
        [*CHECKPOINT_ID_COLUMNS, "profile_id"], dropna=False
    ):
        if len(group) != 2 * len(EXPECTED_WEIGHT_LAYERS):
            continue
        valid = True
        for layer in EXPECTED_WEIGHT_LAYERS:
            rows = group[group["layer"].astype(str) == str(layer)]
            if len(rows) != 2 or set(rows["fit_variant"].astype(str)) != {
                "raw", "clip_xmax"
            }:
                valid = False
                break
            if len(set(rows["transformed_spectrum_sha256"].fillna("").astype(str))) != 1:
                valid = False
                break
        if valid:
            complete_fit_keys.add(
                (str(identity[0]), int(identity[1]), int(identity[2]),
                 int(identity[3]), str(identity[4]))
            )
    complete_secondary_keys = {
        (str(identity[0]), int(identity[1]), int(identity[2]),
         int(identity[3]), str(identity[4]))
        for identity, group in secondary.groupby(
            [*CHECKPOINT_ID_COLUMNS, "profile_id"], dropna=False
        )
        if len(group) == len(EXPECTED_WEIGHT_LAYERS)
        and set(group["layer"].astype(str)) == set(EXPECTED_WEIGHT_LAYERS)
    }
    complete = complete_fit_keys & complete_secondary_keys
    fits = fits.loc[
        [checkpoint_profile_identity(row) in complete for _, row in fits.iterrows()]
    ].copy()
    secondary = secondary.loc[
        [
            checkpoint_profile_identity(row) in complete
            for _, row in secondary.iterrows()
        ]
    ].copy()
    if complete:
        print(
            f"Resuming {method}: {len(complete)} complete checkpoint/profile groups"
        )
    return fits, secondary, complete


def selected_refs(seed_dir):
    refs = tuple(
        checkpoint_refs_at_epoch_stride(
            seed_dir, epoch_stride=ANALYSIS_EPOCH_STRIDE
        )
    )
    maximum = int(MAXIMUM_CHECKPOINTS)
    if maximum < 1 or maximum > 100:
        raise ValueError("MAXIMUM_CHECKPOINTS must lie in [1,100]")
    return refs[-min(maximum, len(refs)):]


def build_checkpoint_contexts():
    rows = []
    for optimizer in OPTIMIZER_SLUGS:
        for seed in ACTIVE_SEEDS:
            seed_dir = require_tail_checkpoint_cache(optimizer, seed)
            fingerprint = verified_run_fingerprint(optimizer, seed)
            refs = selected_refs(seed_dir)
            if (
                int(ANALYSIS_EPOCH_STRIDE) == 1
                and int(MAXIMUM_CHECKPOINTS) == 100
                and len(refs) != 100
            ):
                raise RuntimeError(
                    f"Requested complete final-100 cache, observed {len(refs)} for "
                    f"optimizer={optimizer}, seed={seed}"
                )
            for index, ref in enumerate(refs):
                rows.append({
                    "optimizer": str(optimizer),
                    "seed": int(seed),
                    "epoch": int(ref.epoch),
                    "global_step": int(ref.global_step),
                    "checkpoint_path": str(Path(ref.path).resolve()),
                    "protocol_fingerprint": str(fingerprint),
                    "checkpoint_index": int(index),
                    "checkpoint_count": int(len(refs)),
                    "is_anchor": bool(index == 0),
                    "is_final": bool(index == len(refs) - 1),
                    "source_artifact_kind": "verified_final_100_tail_cache",
                    "analysis_code_sha256": WEIGHT_QUOTIENT_ANALYSIS_CODE_SHA256,
                    "analysis_settings_sha256": (
                        WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_SHA256
                    ),
                    "analysis_settings_json": (
                        WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_JSON
                    ),
                })
    frame = pd.DataFrame(rows).sort_values(
        ["optimizer", "seed", "epoch", "global_step"]
    ).reset_index(drop=True)
    if frame.empty:
        raise RuntimeError("No verified checkpoint contexts were constructed")
    return frame


def active_run_manifests():
    manifests = []
    for optimizer in OPTIMIZER_SLUGS:
        for seed in ACTIVE_SEEDS:
            identity = _VERIFIED_RUN_IDENTITIES.get((str(optimizer), int(seed)))
            if identity is None:
                raise RuntimeError(
                    f"Run identity was not verified for {optimizer}/seed_{seed}"
                )
            source = Path(identity["source_seed_dir"]) / "manifest.json"
            manifests.append(json.loads(source.read_text(encoding="utf-8")))
    return manifests


def run_raw_weightwatcher_controls(contexts):
    fit_path = QUOTIENT_ANALYSIS_DIR / "raw_weightwatcher_dual_fits.csv"
    midpoint_path = QUOTIENT_ANALYSIS_DIR / "midpoint_ecs_ranks.csv"
    raw_profiles = ({"profile_id": "raw_full_weight"},)
    resumed_fits, resumed_midpoints, complete = load_resumable_tables(
        fit_path=fit_path,
        secondary_path=midpoint_path,
        contexts=contexts,
        method="raw_weight_control",
        profiles=raw_profiles,
    )
    fit_frames = [resumed_fits] if not resumed_fits.empty else []
    midpoint_frames = [resumed_midpoints] if not resumed_midpoints.empty else []
    for position, context in contexts.iterrows():
        resume_key = checkpoint_identity(context) + ("raw_full_weight",)
        if resume_key in complete:
            continue
        model = checkpoint_model(context["checkpoint_path"])
        metadata = {
            **context.to_dict(),
            "method": "raw_weight_control",
            "profile_id": "raw_full_weight",
            "parameter_profile": "{}",
            "operator_kind": "weight_esd",
            "map_definition": "identity W -> W",
            "selection_role": "assumption_free_raw_control",
            "randomized_diagnostics_interpretation": (
                "original_matrix_entry_shuffle_baseline_audit"
            ),
        }
        measurement = analyze_weightwatcher_dual(
            model,
            min_evals=int(WW_MIN_EVALS),
            max_evals=WW_MAX_EVALS,
            max_fingers=int(WW_MAX_FINGERS),
            svd_method=str(WW_SVD_METHOD),
            randomize=True,
            analysis_seed=stable_analysis_seed(
                context["optimizer"], context["seed"], context["epoch"], "raw"
            ),
            primary_variant="clip_xmax",
        )
        validation = validate_weightwatcher_measurement(
            measurement,
            primary_variant="clip_xmax",
            expected_layers=EXPECTED_WEIGHT_LAYERS,
        )
        metadata["weightwatcher_structural_errors"] = json.dumps(
            list(validation.structural_errors)
        )
        metadata["weightwatcher_primary_fit_failures"] = json.dumps(
            list(validation.primary_fit_failures)
        )
        hashes = {
            layer: spectrum_sha256(measurement.esds[layer])
            for layer in EXPECTED_WEIGHT_LAYERS
        }
        fit_frames.append(
            weightwatcher_rows(measurement, metadata=metadata, spectrum_hashes=hashes)
        )
        checkpoint_midpoints = []
        for layer in EXPECTED_WEIGHT_LAYERS:
            record = midpoint_record(
                measurement,
                layer=layer,
                metadata=metadata,
            )
            record["final_materialized_gram_eigenvalues_json"] = (
                json.dumps(
                    _jsonable(np.asarray(measurement.esds[layer], dtype=float))
                )
                if bool(context["is_final"])
                else ""
            )
            checkpoint_midpoints.append(record)
        midpoint_frames.append(pd.DataFrame(checkpoint_midpoints))
        atomic_csv(pd.concat(fit_frames, ignore_index=True, sort=False), fit_path)
        atomic_csv(
            pd.concat(midpoint_frames, ignore_index=True, sort=False), midpoint_path
        )
        if (position + 1) % 10 == 0 or position + 1 == len(contexts):
            print(f"raw WeightWatcher: {position + 1}/{len(contexts)} checkpoints")
    fits = pd.concat(fit_frames, ignore_index=True, sort=False)
    midpoints = pd.concat(midpoint_frames, ignore_index=True, sort=False)
    atomic_csv(fits, fit_path)
    atomic_csv(midpoints, midpoint_path)
    final_spectra = {}
    final_spectrum_index = []
    final_rows = midpoints[
        midpoints["is_final"].astype(str).str.lower().eq("true")
    ]
    for _, row in final_rows.iterrows():
        payload = str(row.get("final_materialized_gram_eigenvalues_json", ""))
        if not payload or payload.lower() == "nan":
            raise RuntimeError("Resumable raw final spectrum payload is missing")
        values = np.asarray(json.loads(payload), dtype=float)
        key = stable_slug(
            f"{row['optimizer']}_{int(row['seed'])}_{row['layer']}_raw_weight_control"
        )
        final_spectra[key] = values
        final_spectrum_index.append({
            "spectrum_key": key,
            "optimizer": str(row["optimizer"]),
            "seed": int(row["seed"]),
            "epoch": int(row["epoch"]),
            "global_step": int(row["global_step"]),
            "layer": str(row["layer"]),
            "method": "raw_weight_control",
            "profile_id": "raw_full_weight",
        })
    atomic_npz(
        final_spectra,
        QUOTIENT_ANALYSIS_DIR / "raw_weight_final_spectra.npz",
    )
    atomic_csv(
        pd.DataFrame(final_spectrum_index, columns=FINAL_SPECTRUM_INDEX_COLUMNS),
        QUOTIENT_ANALYSIS_DIR / "raw_weight_final_spectra_index.csv",
    )
    return fits, midpoints


def _midpoint_lookup(midpoints):
    keys = ["optimizer", "seed", "epoch", "global_step", "layer"]
    if midpoints.duplicated(keys).any():
        raise RuntimeError("Midpoint table contains duplicate checkpoint/layer rows")
    return {
        tuple(row[key] for key in keys): int(row["k_mid"])
        for _, row in midpoints.iterrows()
    }


def _anchor_paths(contexts):
    anchors = contexts[contexts["is_anchor"].astype(bool)]
    if anchors.duplicated(["optimizer", "seed"]).any():
        raise RuntimeError("Multiple anchor checkpoints found for one run")
    return {
        (str(row["optimizer"]), int(row["seed"])): str(row["checkpoint_path"])
        for _, row in anchors.iterrows()
    }


def run_quotient_method(method, profiles, *, contexts, midpoints):
    method = str(method)
    if method not in set(EXPECTED_QUOTIENT_METHODS) | set(REFERENCE_METHODS):
        raise ValueError(f"Undeclared quotient method {method!r}")
    profiles = tuple(dict(profile) for profile in profiles)
    profile_ids = tuple(str(profile.get("profile_id", "")) for profile in profiles)
    if not profiles or any(not value for value in profile_ids):
        raise ValueError(f"{method} has an empty or unnamed parameter profile")
    if len(set(profile_ids)) != len(profile_ids):
        raise ValueError(f"{method} has duplicate profile IDs: {profile_ids}")
    if method in METHOD_PROFILE_IDS:
        raise RuntimeError(f"{method} was already run in this notebook")
    METHOD_PROFILE_IDS[method] = profile_ids
    lookup = _midpoint_lookup(midpoints)
    anchors = _anchor_paths(contexts)
    fit_path = QUOTIENT_ANALYSIS_DIR / f"{method}_weightwatcher_dual_fits.csv"
    operator_path = QUOTIENT_ANALYSIS_DIR / f"{method}_operator_rows.csv"
    resumed_fits, resumed_operators, complete = load_resumable_tables(
        fit_path=fit_path,
        secondary_path=operator_path,
        contexts=contexts,
        method=method,
        profiles=profiles,
    )
    fit_frames = [resumed_fits] if not resumed_fits.empty else []
    operator_rows = (
        resumed_operators.to_dict("records") if not resumed_operators.empty else []
    )
    for position, context in contexts.iterrows():
        base_model = None
        anchor_model = None
        for raw_profile in profiles:
            profile_id, options = parameter_profile(raw_profile)
            resume_key = checkpoint_identity(context) + (profile_id,)
            if resume_key in complete:
                continue
            if base_model is None:
                base_model = checkpoint_model(context["checkpoint_path"])
                anchor_model = checkpoint_model(
                    anchors[(str(context["optimizer"]), int(context["seed"]))]
                )
            model = copy.deepcopy(base_model)
            available_layers = []
            unavailable = {}
            hashes = {}
            for layer in EXPECTED_WEIGHT_LAYERS:
                key = (
                    str(context["optimizer"]),
                    int(context["seed"]),
                    int(context["epoch"]),
                    int(context["global_step"]),
                    str(layer),
                )
                k = int(lookup[key])
                weight = matrix_from_model(base_model, layer)
                anchor_weight = matrix_from_model(anchor_model, layer)
                common = {
                    **context.to_dict(),
                    "method": method,
                    "profile_id": profile_id,
                    "layer": str(layer),
                    "ecs_rank": k,
                    "selection_role": "declared_quotient_hypothesis",
                    "parameter_profile": json.dumps(_jsonable(options), sort_keys=True),
                    "randomized_diagnostics_interpretation": (
                        "gauge_dependent_entry_shuffle_audit_not_a_quotient_invariant"
                    ),
                }
                try:
                    result = weight_quotients.apply_weight_quotient(
                        method,
                        weight,
                        ecs_rank=k,
                        anchor_weight=anchor_weight,
                        parameters=options,
                    )
                    if int(result.retained_rank) < int(WW_MIN_EVALS):
                        raise weight_quotients.WeightQuotientUnavailable(
                            f"retained rank {result.retained_rank} is below "
                            f"WW_MIN_EVALS={WW_MIN_EVALS}"
                        )
                    replace_model_matrix(model, layer, result.weight)
                    actual_esd = model_layer_esd(model, layer)
                    declared = np.sort(
                        np.asarray(result.gram_eigenvalues, dtype=float)
                    )[::-1]
                    observed = actual_esd[::-1]
                    if observed.size != declared.size or not np.allclose(
                        observed, declared,
                        rtol=5.0e-5, atol=1.0e-10,
                    ):
                        raise RuntimeError(
                            "float32 canonical matrix disagrees with the declared "
                            "transformed ECS spectrum or retained rank"
                        )
                    digest = spectrum_sha256(actual_esd)
                    hashes[layer] = digest
                    available_layers.append(layer)
                    operator_rows.append({
                        **common,
                        "operator_kind": result.operator_kind,
                        "map_definition": result.map_definition,
                        "retained_rank": int(result.retained_rank),
                        "available": True,
                        "unavailable_reason": "",
                        "materialized_weight_dtype": str(
                            dict(model.named_parameters())[layer].dtype
                        ),
                        "materialized_positive_esd_count": int(actual_esd.size),
                        "materialized_orthogonal_gauge": (
                            "rectangular_diagonal_canonical_section"
                        ),
                        "quotient_parameters": json.dumps(
                            _jsonable(result.parameters), sort_keys=True
                        ),
                        "transformed_spectrum_sha256": digest,
                        "final_materialized_gram_eigenvalues_json": (
                            json.dumps(_jsonable(actual_esd))
                            if bool(context["is_final"])
                            else ""
                        ),
                    })
                except weight_quotients.WeightQuotientUnavailable as error:
                    unavailable[layer] = f"{type(error).__name__}: {error}"
                    operator_rows.append({
                        **common,
                        "operator_kind": f"{method}_unavailable",
                        "map_definition": "declared quotient could not produce a valid WW matrix",
                        "retained_rank": 0,
                        "available": False,
                        "unavailable_reason": unavailable[layer],
                        "quotient_parameters": json.dumps(_jsonable(options), sort_keys=True),
                        "transformed_spectrum_sha256": "",
                        "final_materialized_gram_eigenvalues_json": "",
                    })
            fit_metadata = {
                **context.to_dict(),
                "method": method,
                "profile_id": profile_id,
                "selection_role": "declared_quotient_hypothesis",
                "parameter_profile": json.dumps(_jsonable(options), sort_keys=True),
                "randomized_diagnostics_interpretation": (
                    "gauge_dependent_entry_shuffle_audit_not_a_quotient_invariant"
                ),
            }
            if available_layers:
                measurement = analyze_weightwatcher_dual(
                    model,
                    min_evals=int(WW_MIN_EVALS),
                    max_evals=WW_MAX_EVALS,
                    max_fingers=int(WW_MAX_FINGERS),
                    svd_method=str(WW_SVD_METHOD),
                    randomize=True,
                    analysis_seed=stable_analysis_seed(
                        context["optimizer"], context["seed"], context["epoch"],
                        method, profile_id,
                    ),
                    primary_variant="clip_xmax",
                )
                validation = validate_weightwatcher_measurement(
                    measurement,
                    primary_variant="clip_xmax",
                    expected_layers=EXPECTED_WEIGHT_LAYERS,
                )
                if validation.structural_errors:
                    raise RuntimeError(
                        "Transformed WeightWatcher acquisition is structurally "
                        f"invalid: {validation.structural_errors}"
                    )
                measured = weightwatcher_rows(
                    measurement,
                    metadata=fit_metadata,
                    spectrum_hashes=hashes,
                )
                measured["weightwatcher_primary_fit_failures"] = json.dumps(
                    list(validation.primary_fit_failures)
                )
                measured["weightwatcher_raw_audit_warnings"] = json.dumps(
                    list(validation.raw_audit_warnings)
                )
                for layer in available_layers:
                    observed_digest = spectrum_sha256(measurement.esds[layer])
                    if observed_digest != hashes[layer]:
                        raise RuntimeError(
                            f"WeightWatcher analyzed a different ESD for {layer}: "
                            f"expected={hashes[layer]}, observed={observed_digest}"
                        )
                measured = measured[measured["layer"].isin(available_layers)].copy()
                fit_frames.append(measured)
            for layer, reason in unavailable.items():
                fit_frames.append(
                    unavailable_fit_rows(
                        metadata=fit_metadata,
                        layer=layer,
                        reason=reason,
                    )
                )
            atomic_csv(
                pd.concat(fit_frames, ignore_index=True, sort=False),
                fit_path,
            )
            atomic_csv(pd.DataFrame(operator_rows), operator_path)
        if (position + 1) % 10 == 0 or position + 1 == len(contexts):
            print(f"{method}: {position + 1}/{len(contexts)} checkpoints")
    fits = pd.concat(fit_frames, ignore_index=True, sort=False)
    operators = pd.DataFrame(operator_rows)
    atomic_csv(fits, fit_path)
    atomic_csv(operators, operator_path)
    final_spectra = {}
    final_spectrum_index = []
    final_rows = operators[
        operators["is_final"].astype(str).str.lower().eq("true")
        & operators["available"].astype(str).str.lower().eq("true")
    ]
    for _, row in final_rows.iterrows():
        payload = str(row.get("final_materialized_gram_eigenvalues_json", ""))
        if not payload or payload.lower() == "nan":
            raise RuntimeError(
                f"Resumable final spectrum payload is missing for {method}"
            )
        values = np.asarray(json.loads(payload), dtype=float)
        spectrum_key = stable_slug(
            f"{row['optimizer']}_{int(row['seed'])}_{row['layer']}_{method}_{row['profile_id']}"
        )
        final_spectra[spectrum_key] = values
        final_spectrum_index.append({
            "spectrum_key": spectrum_key,
            "optimizer": str(row["optimizer"]),
            "seed": int(row["seed"]),
            "epoch": int(row["epoch"]),
            "global_step": int(row["global_step"]),
            "layer": str(row["layer"]),
            "method": method,
            "profile_id": str(row["profile_id"]),
        })
    # Always overwrite both files. An empty current result must not fall back
    # to a stale archive from an earlier execution in the same output folder.
    atomic_npz(
        final_spectra,
        QUOTIENT_ANALYSIS_DIR / f"{method}_final_spectra.npz",
    )
    atomic_csv(
        pd.DataFrame(final_spectrum_index, columns=FINAL_SPECTRUM_INDEX_COLUMNS),
        QUOTIENT_ANALYSIS_DIR / f"{method}_final_spectra_index.csv",
    )
    METHOD_FIT_FRAMES[method] = fits
    METHOD_OPERATOR_FRAMES[method] = operators
    return fits, operators


In [ ]:
RUN_VARIANT = 'one_seed'
METHOD_SLUG = "weight_only_muon_quotients_" + RUN_VARIANT
UNCERTAINTY_POLICY = 'no_seed_error_bars'
NOTEBOOK_BUILDER_SOURCE_SHA256 = 'c790ee35d32dea834c9063c5a7ecaaa06e767127944138dd9fbc17edfef2b39c'
WEIGHT_QUOTIENT_ANALYSIS_CODE_SHA256 = hashlib.sha256(
    (
        WEIGHT_QUOTIENT_ANALYSIS_VERSION
        + WEIGHT_QUOTIENT_RUNTIME_DEPENDENCY_SHA256
        + NOTEBOOK_BUILDER_SOURCE_SHA256
    ).encode("utf-8")
).hexdigest()
OPTIMIZER_SLUGS = [
    str(optimizer)
    for optimizer in normalize_papermill_sequence(
        OPTIMIZER_SLUGS, name="OPTIMIZER_SLUGS"
    )
]
LAYERS = [
    str(layer)
    for layer in normalize_papermill_sequence(LAYERS, name="LAYERS")
]
RUN_PARAMETER_SCANS = normalize_papermill_bool(
    RUN_PARAMETER_SCANS, name="RUN_PARAMETER_SCANS"
)
RESUME_PARTIAL_RESULTS = normalize_papermill_bool(
    RESUME_PARTIAL_RESULTS, name="RESUME_PARTIAL_RESULTS"
)

ACTIVE_SEEDS = (int(SEED),)
if len(ACTIVE_SEEDS) != 1:
    raise RuntimeError("The one-seed notebook must analyze exactly one complete run")

if tuple(LAYERS) != EXPECTED_WEIGHT_LAYERS:
    raise ValueError(f"LAYERS must remain {EXPECTED_WEIGHT_LAYERS}; observed {LAYERS}")
QUOTIENT_ANALYSIS_DIR = OUTPUT_ROOT_PATH / METHOD_SLUG
QUOTIENT_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
atomic_json(
    {
        "schema_version": 1,
        "method_slug": METHOD_SLUG,
        "run_variant": RUN_VARIANT,
        "completed": False,
        "analysis_version": WEIGHT_QUOTIENT_ANALYSIS_VERSION,
        "analysis_code_sha256": WEIGHT_QUOTIENT_ANALYSIS_CODE_SHA256,
        "analysis_settings_sha256": WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_SHA256,
        "analysis_settings": WEIGHT_QUOTIENT_ANALYSIS_SETTINGS,
        "notebook_builder_source_sha256": NOTEBOOK_BUILDER_SOURCE_SHA256,
        "status": "analysis_in_progress_or_interrupted",
    },
    QUOTIENT_ANALYSIS_DIR / "method_provenance.json",
)
METHOD_FIT_FRAMES = {}
METHOD_OPERATOR_FRAMES = {}
METHOD_PROFILE_IDS = {}

checkpoint_contexts = build_checkpoint_contexts()

CROSS_RUN_PROVENANCE_AUDITED = False

raw_fit_rows, midpoint_rows = run_raw_weightwatcher_controls(checkpoint_contexts)
display(checkpoint_contexts.head())
display(midpoint_rows.head())
print("uncertainty policy:", UNCERTAINTY_POLICY)
print("quotient outputs:", QUOTIENT_ANALYSIS_DIR)


## Reference controls: midpoint truncation and the polar singular shift

The raw full weight is already saved above. The first reference is
pure midpoint-ECS truncation. The second is the previously proposed
one-coordinate polar quotient

\[
\lambda_i'=(\sqrt{\lambda_i}-\mu)^2.
\]

These references are not counted among the five alternative cells.


In [ ]:
midpoint_control_fits, midpoint_control_operators = run_quotient_method(
    "midpoint_ecs_control",
    [{"profile_id": "midpoint_ecs_no_counterterm"}],
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
uniform_singular_fits, uniform_singular_operators = run_quotient_method(
    "uniform_singular_translation",
    choose_profiles(UNIFORM_SINGULAR_PRIMARY, UNIFORM_SINGULAR_SCAN),
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
display(uniform_singular_fits.head())


## 1. Gram ridge counterterm

Test the isotropic covariance hypothesis

\[
\lambda_i'=(\lambda_i-\tau)_+.
\]

This is exact only for a scalar Gram contribution. The full scan is
preserved; no candidate is selected by proximity to \(\alpha=2\).


In [ ]:
gram_ridge_fits, gram_ridge_operators = run_quotient_method(
    "gram_ridge",
    choose_profiles(GRAM_RIDGE_PRIMARY, GRAM_RIDGE_SCAN),
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
display(gram_ridge_fits.head())


## 2. Blockwise singular counterterms

Split the ordered midpoint ECS into a fixed number of contiguous
bands and apply \(s_i'=(s_i-\mu_b)_+\). A declared PAVA projection
restores non-increasing order if a block shift creates crossings;
its correction norm is recorded.


In [ ]:
blockwise_fits, blockwise_operators = run_quotient_method(
    "blockwise_singular",
    choose_profiles(BLOCKWISE_PRIMARY, BLOCKWISE_SCAN),
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
display(blockwise_fits.head())


## 3. Anchor-frozen Feshbach downfolding

Freeze the smaller-Gram basis at the earliest verified tail
checkpoint of the same optimizer/seed/layer, then evaluate

\[
H_{\mathrm{eff}}(-\delta)=A-B(C+\delta I)^{-1}B^T.
\]

The solve condition number, residual and coupling norm are saved.
At the anchor itself, \(B=0\) and this must reduce exactly to ECS
truncation. Later nonzero coupling measures rotation relative to
that anchor; this is therefore trajectory-dependent, not a strict
single-checkpoint quotient.


In [ ]:
feshbach_fits, feshbach_operators = run_quotient_method(
    "feshbach_downfolding",
    choose_profiles(FESHBACH_PRIMARY, FESHBACH_SCAN),
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
display(feshbach_operators.head())


## 4. Rectangular empirical D-transform deconvolution

Estimate the rectangular noise law from a declared lower fraction
of the discarded singular modes and map a retained spike using
\(D(s)^{-1/2}\). This is an
empirical separated-spike approximation to the incoherent/free
additive-noise hypothesis, not an unrestricted full-rank free
deconvolution solver. The scan varies the operative bulk window;
rows lacking eight selected noise modes or a separated edge remain
explicitly unavailable.


In [ ]:
rectangular_d_fits, rectangular_d_operators = run_quotient_method(
    "rectangular_d_transform",
    choose_profiles(RECTANGULAR_D_PRIMARY, RECTANGULAR_D_SCAN),
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
display(rectangular_d_operators.head())


## 5. Calibrated monotone MP shrinker

Calibrate a white-noise scale from the discarded spectral edge and
apply the analytic monotone optimal Frobenius shrinker. The
calibration never uses WeightWatcher \(\alpha\), KS distance,
`xmin`, fingers or trace-log. This is the reproducible baseline for
a later shrinker calibrated on recorded Muon corruptions.


In [ ]:
calibrated_fits, calibrated_operators = run_quotient_method(
    "calibrated_mp_shrinker",
    choose_profiles(CALIBRATED_SHRINKER_PRIMARY, CALIBRATED_SHRINKER_SCAN),
    contexts=checkpoint_contexts,
    midpoints=midpoint_rows,
)
display(calibrated_operators.head())


## Exact-grid and dual-WeightWatcher audit

Consolidate every row, verify that the five declared methods and
both references ran, and prove that raw/fixed-finger fits share the
identical transformed spectrum hash.


In [ ]:
expected_methods = set(EXPECTED_QUOTIENT_METHODS) | set(REFERENCE_METHODS)
if set(METHOD_FIT_FRAMES) != expected_methods:
    raise RuntimeError(
        f"Quotient inventory drift: observed={sorted(METHOD_FIT_FRAMES)}, "
        f"expected={sorted(expected_methods)}"
    )
quotient_fit_rows = pd.concat(
    [raw_fit_rows, *METHOD_FIT_FRAMES.values()], ignore_index=True, sort=False
)
quotient_operator_rows = pd.concat(
    list(METHOD_OPERATOR_FRAMES.values()), ignore_index=True, sort=False
)
pair_keys = [
    "optimizer", "seed", "epoch", "global_step", "method", "profile_id", "layer"
]
for identity, group in quotient_fit_rows.groupby(pair_keys, dropna=False):
    variants = set(group["fit_variant"].astype(str))
    if variants != {"raw", "clip_xmax"}:
        raise RuntimeError(f"Incomplete dual WeightWatcher pair {identity}: {variants}")
    hashes = set(group["transformed_spectrum_sha256"].fillna("").astype(str))
    if len(hashes) != 1:
        raise RuntimeError(
            f"Raw and clip_xmax fits used different transformed spectra: {identity}"
        )
grid_columns = pair_keys + ["fit_variant"]
if quotient_fit_rows.duplicated(grid_columns).any():
    duplicates = quotient_fit_rows.loc[
        quotient_fit_rows.duplicated(grid_columns, keep=False), grid_columns
    ].head(20)
    raise RuntimeError(f"Duplicate quotient fit rows:\n{duplicates}")
expected_profile_ids = {
    "raw_weight_control": ("raw_full_weight",),
    **METHOD_PROFILE_IDS,
}
observed_methods = set(quotient_fit_rows["method"].astype(str))
if observed_methods != set(expected_profile_ids):
    raise RuntimeError(
        f"Fit method inventory drift: observed={sorted(observed_methods)}, "
        f"expected={sorted(expected_profile_ids)}"
    )
expected_grid = {
    (
        str(context["optimizer"]), int(context["seed"]), int(context["epoch"]),
        int(context["global_step"]), str(method), str(profile_id), str(layer),
        str(fit_variant),
    )
    for _, context in checkpoint_contexts.iterrows()
    for method, profile_ids in expected_profile_ids.items()
    for profile_id in profile_ids
    for layer in EXPECTED_WEIGHT_LAYERS
    for fit_variant in ("raw", "clip_xmax")
}
observed_grid = {
    (
        str(row.optimizer), int(row.seed), int(row.epoch), int(row.global_step),
        str(row.method), str(row.profile_id), str(row.layer), str(row.fit_variant),
    )
    for row in quotient_fit_rows[grid_columns].itertuples(index=False)
}
if observed_grid != expected_grid:
    raise RuntimeError(
        f"Incomplete quotient grid: missing={len(expected_grid-observed_grid)}, "
        f"unexpected={len(observed_grid-expected_grid)}"
    )
atomic_csv(
    quotient_fit_rows,
    QUOTIENT_ANALYSIS_DIR / "all_weightwatcher_dual_fits.csv",
)
atomic_csv(
    quotient_operator_rows,
    QUOTIENT_ANALYSIS_DIR / "all_quotient_operator_rows.csv",
)
availability_summary = (
    quotient_operator_rows.assign(
        available=truthy_series(quotient_operator_rows["available"])
    )
    .groupby(["optimizer", "method", "profile_id", "layer"], as_index=False)
    .agg(
        available_checkpoint_count=("available", "sum"),
        expected_checkpoint_count=("available", "size"),
        availability_fraction=("available", "mean"),
    )
)
atomic_csv(
    availability_summary,
    QUOTIENT_ANALYSIS_DIR / "quotient_availability_summary.csv",
)
manifest = {
    "schema_version": 1,
    "completed": False,
    "status": "exact_grid_validated_reporting_in_progress",
    "suite_name": PROTOCOL_SLUG,
    "method_slug": METHOD_SLUG,
    "analysis_version": WEIGHT_QUOTIENT_ANALYSIS_VERSION,
    "analysis_code_sha256": WEIGHT_QUOTIENT_ANALYSIS_CODE_SHA256,
    "analysis_settings_sha256": WEIGHT_QUOTIENT_ANALYSIS_SETTINGS_SHA256,
    "analysis_settings": WEIGHT_QUOTIENT_ANALYSIS_SETTINGS,
    "notebook_builder_source_sha256": NOTEBOOK_BUILDER_SOURCE_SHA256,
    "run_variant": RUN_VARIANT,
    "uncertainty_policy": UNCERTAINTY_POLICY,
    "cross_run_provenance_audited": bool(CROSS_RUN_PROVENANCE_AUDITED),
    "optimizer_slugs": list(OPTIMIZER_SLUGS),
    "seeds": list(ACTIVE_SEEDS),
    "optimizer_seed_protocol_fingerprints": {
        f"{row.optimizer}/seed_{int(row.seed)}": str(row.protocol_fingerprint)
        for row in checkpoint_contexts[
            ["optimizer", "seed", "protocol_fingerprint"]
        ].drop_duplicates().itertuples(index=False)
    },
    "source_artifact_kinds": sorted(
        checkpoint_contexts["source_artifact_kind"].astype(str).unique().tolist()
    ),
    "checkpoint_count_per_run": int(
        checkpoint_contexts.groupby(["optimizer", "seed"]).size().min()
    ),
    "analysis_epoch_stride": int(ANALYSIS_EPOCH_STRIDE),
    "layers": list(EXPECTED_WEIGHT_LAYERS),
    "five_quotient_methods": list(EXPECTED_QUOTIENT_METHODS),
    "reference_methods": list(REFERENCE_METHODS),
    "method_profile_ids": {
        method: list(profile_ids)
        for method, profile_ids in METHOD_PROFILE_IDS.items()
    },
    "weightwatcher_fit_variants": ["raw", "clip_xmax"],
    "fixed_finger_policy": "fix_fingers=clip_xmax",
    "resume_partial_results": bool(RESUME_PARTIAL_RESULTS),
    "orthogonal_quotient_group": "O(out) x O(in)",
    "canonical_section": "rectangular_diagonal",
    "randomized_diagnostics_policy": (
        "computed_for_baseline_API_compatibility_but_gauge_dependent_and_not_"
        "interpreted_as_quotient_invariants"
    ),
    "run_parameter_scans": bool(RUN_PARAMETER_SCANS),
    "midpoint_selected_before_quotient": True,
    "global_rescaling_used_to_change_alpha": False,
    "method_claim": "falsifiable_weight_only_inverse_models_not_exact_unknown_Muon_quotient",
}
atomic_json(manifest, QUOTIENT_ANALYSIS_DIR / "method_provenance.json")
display(quotient_fit_rows.head(30))
display(quotient_operator_rows.head(30))


## One-seed trajectories

Plot the complete checkpoint trajectory without confidence
intervals. Checkpoints, layers, modes and finger choices are
not independent replicates and cannot create error bars.


In [ ]:
one_seed = quotient_fit_rows.copy()
one_seed["alpha"] = pd.to_numeric(one_seed["alpha"], errors="coerce")
one_seed["fit_success"] = truthy_series(one_seed["fit_ok"]) & np.isfinite(one_seed["alpha"])
atomic_csv(
    one_seed[~one_seed["fit_success"]].copy(),
    QUOTIENT_ANALYSIS_DIR / "failed_or_unavailable_one_seed_fits.csv",
)
one_seed = one_seed[one_seed["fit_success"]].copy()
for (optimizer, layer, fit_variant), panel in one_seed.groupby(
    ["optimizer", "layer", "fit_variant"]
):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for (method, profile_id), curve in panel.groupby(["method", "profile_id"]):
        curve = curve.sort_values("epoch")
        ax.plot(
            curve["epoch"], curve["alpha"], linewidth=1.4,
            label=f"{method}:{profile_id}",
        )
    ax.axhline(2.0, color="#222222", linestyle="--", linewidth=1.2)
    ax.set(
        xlabel="epoch",
        ylabel="WeightWatcher alpha",
        title=f"{optimizer} {layer}: {fit_variant} quotient alpha (one seed; no CI)",
    )
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=6, ncol=2)
    fig.tight_layout()
    fig.savefig(
        QUOTIENT_ANALYSIS_DIR
        / f"alpha_one_seed_{stable_slug(optimizer)}_{stable_slug(layer)}_{stable_slug(fit_variant)}.png",
        dpi=180,
        bbox_inches="tight",
    )
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
print("No error bars were computed:", UNCERTAINTY_POLICY)


## Final-checkpoint spectral-density gallery

Plot the empirical CCDFs of the exact Gram eigenvalues supplied
to WeightWatcher. This is an ESD visualization, not an extra fit.
Both raw and `clip_xmax` fit rows reference these same spectra.


In [ ]:
spectrum_rows = []
spectrum_stems = ["raw_weight_final_spectra"] + [
    f"{method}_final_spectra" for method in sorted(METHOD_FIT_FRAMES)
]
for stem in spectrum_stems:
    index_path = require_path(
        QUOTIENT_ANALYSIS_DIR / f"{stem}_index.csv",
        description=f"{stem} index",
    )
    archive_path = QUOTIENT_ANALYSIS_DIR / f"{stem}.npz"
    archive = np.load(require_path(archive_path, description="final spectrum archive"))
    index = pd.read_csv(index_path)
    for _, row in index.iterrows():
        values = np.asarray(archive[str(row["spectrum_key"])], dtype=float)
        values = np.sort(values[np.isfinite(values) & (values > 0.0)])
        spectrum_rows.append({**row.to_dict(), "values": values})
final_spectrum_index = pd.DataFrame(
    [{key: value for key, value in row.items() if key != "values"} for row in spectrum_rows]
)
atomic_csv(final_spectrum_index, QUOTIENT_ANALYSIS_DIR / "all_final_spectra_index.csv")

for optimizer in OPTIMIZER_SLUGS:
    for layer in EXPECTED_WEIGHT_LAYERS:
        selected = [
            row for row in spectrum_rows
            if str(row["optimizer"]) == str(optimizer) and str(row["layer"]) == str(layer)
        ]
        if not selected:
            continue
        fig, ax = plt.subplots(figsize=(9.5, 6.0))
        for row in selected:
            values = row["values"]
            ccdf = np.arange(values.size, 0, -1, dtype=float) / values.size
            ax.loglog(
                values,
                ccdf,
                linewidth=1.0,
                alpha=0.70,
                label=f"s{row['seed']} {row['method']}:{row['profile_id']}",
            )
        ax.set(
            xlabel="Gram eigenvalue",
            ylabel="empirical CCDF",
            title=f"{optimizer} {layer}: final quotient ESDs",
        )
        ax.grid(True, which="both", alpha=0.22)
        ax.legend(fontsize=5.5, ncol=2)
        fig.tight_layout()
        fig.savefig(
            QUOTIENT_ANALYSIS_DIR
            / f"final_esd_ccdf_{stable_slug(optimizer)}_{stable_slug(layer)}.png",
            dpi=180,
            bbox_inches="tight",
        )
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)
display(final_spectrum_index.head(30))
manifest["completed"] = True
manifest["status"] = "complete"
manifest["fit_row_count"] = int(len(quotient_fit_rows))
manifest["operator_row_count"] = int(len(quotient_operator_rows))
manifest["final_spectrum_count"] = int(len(final_spectrum_index))
atomic_json(manifest, QUOTIENT_ANALYSIS_DIR / "method_provenance.json")
print("complete quotient analysis:", QUOTIENT_ANALYSIS_DIR)


### Interpretation contract

A lower KS distance or an alpha nearer two does not establish
that a method inverted Muon. Promote a rule only after planted
corruption recovery, non-power-law negative controls, stable
interior parameters, retained-rank checks, and independent-seed
reproduction. The raw weight ESD remains the assumption-free
state observable.
